fig_site — Fig. `fig:site` (single lossy site)
Reads `data/` (default) or a fresh `output/` (`USE_OUTPUT = True`, via `postprocess.build`) and writes `fig_site.pdf`.

In [1]:
USE_OUTPUT = False
OUTPUT_DIR = "output"

In [2]:
import sys
from pathlib import Path
import numpy as np
HERE = Path.cwd()
sys.path.insert(0, str(HERE.parent))
from common import io as IO
from common.style import plt, W2, style, panel, legend_handles, LABEL, save
if USE_OUTPUT:
    import postprocess
    D = postprocess.build(OUTPUT_DIR)
else:
    D = IO.read_data(HERE / "data")
C, E = D["curves.csv"], D["endpoints.csv"]
METH = ["HF", "2B", "TPPX", "mGW", "DSL"]
cur = lambda m, n0: IO.select(C, method=m, n0=n0)

In [3]:
fig, ax = plt.subplots(1, 3, figsize=(W2, 2.35))
fig.subplots_adjust(left=0.07, right=0.975, top=0.80, bottom=0.17, wspace=0.40)
a = ax[0]; n0 = 0.9                                   # (a) n_sigma(gamma t) from n0 = 0.9
for m in ["exact"] + METH:
    c = cur(m, n0); s = c["gt"] > 0
    a.semilogx(c["gt"][s], c["n_up"][s], **style(m), zorder=2 if m == "exact" else (4 if m == "DSL" else 3))
a.set_xlim(3e-2, 2e3); a.set_ylim(0.0, 0.9)          # cut at 0: mGW goes negative and leaves the frame
a.set_xlabel(r"$\gamma t$"); a.set_ylabel(r"$n_\sigma(t)$"); panel(a, "(a)")
a.text(0.97, 0.95, r"$n_0 = 0.9$", transform=a.transAxes, ha="right", va="top")
a = ax[1]; n0 = 0.5                                   # (b) D(gamma t) from n0 = 0.5
for m in ["exact"] + METH:
    c = cur(m, n0); s = c["gt"] > 0
    a.semilogx(c["gt"][s], c["D"][s], **style(m), zorder=2 if m == "exact" else (4 if m == "DSL" else 3))
a.axhline(0, color="#52514e", lw=0.6)
a.set_xlim(3e-2, 1e2); a.set_ylim(-0.08, 0.26)
a.set_xlabel(r"$\gamma t$"); a.set_ylabel(r"$D(t)$"); panel(a, "(b)")
a.text(0.97, 0.95, r"$n_0 = 0.5$", transform=a.transAxes, ha="right", va="top")
a = ax[2]                                             # (c) n_sigma(gamma t = 2000) versus n0
e = IO.select(E, method="exact"); a.plot(e["n0"], e["n_up"], **style("exact"), zorder=2)
for m in METH:
    e = IO.select(E, method=m)
    a.plot(e["n0"], e["n_up"], marker="o", ms=2.2, markevery=2, **style(m), zorder=4 if m == "DSL" else 3)
a.set_xlim(0, 1); a.set_ylim(-0.02, 0.54)
a.set_xlabel(r"$n_0$"); a.set_ylabel(r"$n_\sigma(\gamma t = 2000)$"); panel(a, "(c)")
a.text(0.5, 0.07, r"MF, TPPX $\to 0$ as $t\to\infty$", transform=a.transAxes, ha="center", va="bottom", fontsize=6.5)
M = ["exact"] + METH
fig.legend(legend_handles(M), [LABEL[m] for m in M], loc="upper center", ncol=6, bbox_to_anchor=(0.53, 1.0))
save(fig, HERE / "fig_site")